# Bellabeat — Process Phase

Combine the two FitBit data-collection periods, resolve the overlapping date between them, clean
known data-quality issues from the Prepare phase, and merge daily activity with sleep data.

Source data: `data/raw/2016-03-12_to_2016-04-11/` and `data/raw/2016-04-12_to_2016-05-12/` (not
committed — see `data/raw/README.md`). Output: `data/processed/*.parquet` (not committed —
regenerate by running this notebook).

## Step 1 — Load daily activity from both periods

In [1]:
import pandas as pd
import os

RAW = "../data/raw"
OUT_DIR = "../data/processed"
os.makedirs(OUT_DIR, exist_ok=True)
P1 = os.path.join(RAW, "2016-03-12_to_2016-04-11")
P2 = os.path.join(RAW, "2016-04-12_to_2016-05-12")

d1 = pd.read_csv(os.path.join(P1, "dailyActivity_merged.csv"), parse_dates=["ActivityDate"])
d2 = pd.read_csv(os.path.join(P2, "dailyActivity_merged.csv"), parse_dates=["ActivityDate"])
print(f"Period 1: {d1.shape}, Period 2: {d2.shape}")

Period 1: (457, 15), Period 2: (940, 15)

## Step 2 — Resolve the overlapping date (2016-04-12)

Both periods include April 12, 2016 — but with **conflicting values**. Period 1's export appears
to cut off mid-day for that date (e.g. user `1503960366` shows 224 steps in Period 1 vs. 13,162 in
Period 2 for the same date), while Period 2's version captures the full day. Period 2's row is
kept.

In [2]:
overlap_date = pd.Timestamp("2016-04-12")
before = len(d1)
d1 = d1[d1["ActivityDate"] != overlap_date]
print(f"Dropped {before - len(d1)} Period-1 rows on the overlap date (kept Period 2's version).")

Dropped 24 Period-1 rows on the overlap date (kept Period 2's version).

## Step 3 — Combine into one daily activity table and standardize column names

In [3]:
activity = pd.concat([d1, d2], ignore_index=True)
activity.columns = [
    "id", "activity_date", "total_steps", "total_distance", "tracker_distance",
    "logged_activities_distance", "very_active_distance", "moderately_active_distance",
    "light_active_distance", "sedentary_active_distance", "very_active_minutes",
    "fairly_active_minutes", "lightly_active_minutes", "sedentary_minutes", "calories",
]
dupes = activity.duplicated(subset=["id", "activity_date"]).sum()
print(f"Combined shape: {activity.shape}. Duplicate (id, activity_date) pairs: {dupes}")
print(f"Unique users: {activity['id'].nunique()}")
print(f"Date range: {activity['activity_date'].min().date()} -> {activity['activity_date'].max().date()}")

Combined shape: (1373, 15). Duplicate (id, activity_date) pairs: 0
Unique users: 35
Date range: 2016-03-12 -> 2016-05-12

## Step 4 — Flag likely non-wear days (zero steps)

**Decision:** flag, don't drop. A day with 0 total steps is much more likely to be a tracker not
worn than genuine zero activity — but removing it outright would silently bias activity averages.
Instead it's kept and flagged so the Analyze phase can report figures both ways.

In [4]:
activity["is_zero_steps"] = activity["total_steps"] == 0
n_zero = activity["is_zero_steps"].sum()
print(f"{n_zero} / {len(activity)} rows ({n_zero/len(activity)*100:.1f}%) have zero total steps.")

133 / 1373 rows (9.7%) have zero total steps.

## Step 5 — Clean sleep data

Only the second period ships a `sleepDay_merged.csv` table — the first period has no daily-level
sleep aggregate. 3 exact duplicate rows are dropped, and a `sleep_efficiency_pct` column is added
(minutes asleep ÷ minutes in bed).

In [5]:
sleep = pd.read_csv(os.path.join(P2, "sleepDay_merged.csv"), parse_dates=["SleepDay"])
before = len(sleep)
sleep = sleep.drop_duplicates()
print(f"Dropped {before - len(sleep)} exact duplicate rows. Remaining: {len(sleep)}")

sleep.columns = ["id", "sleep_day", "total_sleep_records", "total_minutes_asleep", "total_time_in_bed"]
sleep["sleep_efficiency_pct"] = (sleep["total_minutes_asleep"] / sleep["total_time_in_bed"] * 100).round(1)
print(f"Unique users with sleep data: {sleep['id'].nunique()} (out of {activity['id'].nunique()} total)")

Dropped 3 exact duplicate rows. Remaining: 410
Unique users with sleep data: 24 (out of 35 total)

## Step 6 — Merge activity + sleep (left join on id + date)

In [6]:
merged = activity.merge(
    sleep, left_on=["id", "activity_date"], right_on=["id", "sleep_day"], how="left"
)
n_with_sleep = merged["total_minutes_asleep"].notna().sum()
print(f"Merged shape: {merged.shape}. Rows with matching sleep data: {n_with_sleep} "
      f"({n_with_sleep/len(merged)*100:.1f}%)")

Merged shape: (1373, 21). Rows with matching sleep data: 410 (29.9%)

## Step 7 — Save processed tables

Saved as Parquet, excluded from version control (see repo `.gitignore`) — regenerate by
re-running this notebook against `data/raw/`.

In [7]:
activity.to_parquet(os.path.join(OUT_DIR, "daily_activity.parquet"), index=False)
sleep.to_parquet(os.path.join(OUT_DIR, "sleep_day.parquet"), index=False)
merged.to_parquet(os.path.join(OUT_DIR, "activity_sleep_merged.parquet"), index=False)
for f in ["daily_activity.parquet", "sleep_day.parquet", "activity_sleep_merged.parquet"]:
    size_kb = os.path.getsize(os.path.join(OUT_DIR, f)) / 1024
    print(f"Saved {f} ({size_kb:.1f} KB)")

Saved daily_activity.parquet (58.1 KB)
Saved sleep_day.parquet (8.9 KB)
Saved activity_sleep_merged.parquet (66.3 KB)

## Summary

| Step | Result |
|---|---|
| Combined daily activity | 1,373 rows, 35 users, Mar 12 – May 12 2016 |
| Overlap-date conflict | 24 Period-1 rows dropped, Period-2 kept |
| Zero-step days | 133 (9.7%) flagged, not dropped |
| Sleep data | 410 rows, 24 users (69% of the 35), 3 duplicates dropped |
| Activity + sleep merge | 1,373 rows; 29.9% have matching sleep data |

Cross-validated against an equivalent DuckDB SQL pipeline
([`sql/01_process_data.sql`](../sql/01_process_data.sql)) — identical row and user counts.